In [6]:
import numpy as np # used fro basic numerical operations and array clipping
from scipy.stats import binom # Imports the binomial distribution (used to mathematically find our decision boundary based on probability)
from qiskit import QuantumCircuit, transpile #Imports the tools needed to build quantum circuits,  Transpile Converts our circuit into one compatible with the simulator.
from qiskit_aer import AerSimulator #helps simulate quantum circuits
from qiskit_aer.noise import NoiseModel, depolarizing_error #simulates noise and error

_BACKEND = AerSimulator() #Initializes a high-performance local quantum simulator.

n_channels=30


def snr_db_to_ploss(snr_db, floor=0.02, ceiling=0.98): #defines a function with input=SNR and output=channel depolarising probabability
    """Maps sensing SNR to channel depolarizing loss."""
    gamma = 10 ** (snr_db / 10.0) #converts signal to noise ratio from decibles to linear scale
    p_loss = 1.0 / (1.0 + gamma) #maps SNR to depolarizing probability
    return float(np.clip(p_loss, floor, ceiling)) #prevents impossible values (min loss=2% and max loss=98%)


def _quantum_circuit():
    qc = QuantumCircuit(2, 2) #total 4 bits considered, 2 qubits (signal and idler), 2 classical (for storing results)
    qc.h(0)        # Hadamard gate- idler (put in a state of superposition- the state (0 or 1) cannot be determined till measured, exist simulataneously)
    qc.cx(0, 1)    # CNOT gate- entangle idler (0) with signal (1) 
    qc.id(1)       # <- channel noise attaches here
    qc.cx(0, 1)    # receiver - recerses the above 2 operations to check if the 2 photons were ever entangled. 
    qc.h(0)
    qc.measure([0, 1], [0, 1]) #measure both qubits can be 00, 01,10,11
    return qc


def _classical_circuit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)        # single unentangled probe
    qc.id(0)       # <- channel noise attaches here
    qc.h(0)
    qc.measure(0, 0)
    return qc


def _noise_model(p_dep, qubit): #creates noise
    nm = NoiseModel() #empty noise container
    nm.add_quantum_error(depolarizing_error(p_dep, 1), "id", [qubit]) #whenever simulator reaches id, add depolarizing with probability
    return nm


def _run_counts(qc, p_dep, qubit, shots): #runs simulation
    nm = _noise_model(p_dep, qubit) #creates channel
    qc_t = transpile(qc, basis_gates=["h", "cx", "id"], optimization_level=0) #creates the circuit with the given gates- optimization level=0 Do not optimize the circuit. Only make it executable on the backend.
    job = _BACKEND.run(qc_t, shots=shots, noise_model=nm) #runs independent experiments
    return job.result().get_counts() #gives measurement counts ex: 00:73, 01:14, 10:13, 11:78- out of all the times the shots were ran, how many times each outcome came
#shots meaning: prepare qubits, apply gates, apply channel noise, measure, record results

def calibrate_system(strategy, target_pfa, n_window, calibration_shots=1000):
    """
    Step 1: Calibrate the system using a known empty channel (H0) 
    to find the baseline success rate p0, and compute the decision threshold tau.
    """
    #τ is the minimum number of hits required to declare that a channel is occupied, while ensuring that the probability of falsely declaring an empty channel as occupied
    #stays below the chosen limit
    if strategy == "quantum":
        qc = _quantum_circuit()
        target = "00" # correct when channel occupied, signal photon reflected and tested with kept idler, 00=>once entangled
        #The original Bell-state correlation has survived well enough that the decoder reconstructed the initial state correctly.
        qubit = 1
    else:
        qc = _classical_circuit()
        target = "0"
        qubit = 0
        
    # Run a large calibration batch to get a highly accurate p0 baseline
    counts = _run_counts(qc, p_dep=1.0, qubit=qubit, shots=calibration_shots)
    p0 = counts.get(target, 0) / calibration_shots #basline probability: p₀ = the probability of observing the target outcome when the channel is known to be empty (H₀)
    
    # Calculate the smallest threshold tau to satisfy our target Pfa
    tau = n_window + 1 #threshold shots required to get pfa<target pfa
    for t in range(n_window + 1):
        pfa_t = 1.0 if t == 0 else binom.sf(t - 1, n_window, p0) #computes the false alarm (saying occupied when actually free) probability for a given threshold t.
        if pfa_t <= target_pfa:
            tau = t
            break
            
    return tau, p0


def sense_single_channel(strategy, is_occupied, p_loss, n_window, tau):
    """
    Step 2 & 3: Interrogate a channel of unknown/given state.
    Fires exactly n_window shots (pulses) and returns a binary decision.
    """
    # If the channel is occupied, noise is p_loss. If free, it's total noise (1.0).
    p_dep = p_loss if is_occupied else 1.0 #prob of depolarization is=1, if the channel is not there, meaning all noise, complete depolarization, else is=prob of loss
    
    if strategy == "quantum":
        qc = _quantum_circuit()
        target = "00"
        qubit = 1
    else:
        qc = _classical_circuit()
        target = "0"
        qubit = 0
        
    # We transmit exactly `n_window` pulses down the channel
    counts = _run_counts(qc, p_dep=p_dep, qubit=qubit, shots=n_window) #This line runs the quantum (or classical) circuit exactly n_window times
    hits = counts.get(target, 0) #The detector is interested only in the target outcome
    
    # Binary Decision Rule: count must meet or exceed our calibrated threshold
    prediction = 1 if hits >= tau else 0 #This is the detector's decision rule, if hits>=tau, then prediction=1, else 0
    return prediction, hits


def run_spectrum_sensing_simulation(ground_truth, strategy="quantum", snr_db=8.0,
                                    n_window=20, target_pfa=0.05): #sensing experiment done
    """
    Step 4: Sweeps across our channel array, performs active sensing, 
    and compares predictions against the real ground truth.
    """
    p_loss = snr_db_to_ploss(snr_db) #associating snr to loss
    
    print(f"--- Starting Active Sensing Session ({strategy.upper()} strategy) ---")
    print(f"Sensing SNR: {snr_db} dB (Effective p_loss: {p_loss:.3f})")
    print(f"Pulses per channel (N): {n_window} | Max allowed false alarm rate (Pfa): {target_pfa}\n")
    
    # 1. Calibrate system to find decision boundary (tau) 
    tau, p0 = calibrate_system(strategy, target_pfa, n_window) #tells how the empty channel actually looks like
    print(f"[Calibration] Empty-channel success rate (p0): {p0:.3f}")
    print(f"[Calibration] Calculated threshold (tau): Declare Occupied if hits >= {tau} out of {n_window}\n")
    
    predictions = [] #defining empty arrays that will be appended- predictions=[1,0,0,1,0] etc, channel occupied or free
    all_hits = [] #stores the number of hits per channel used for detection
    
    # 2 & 3. Actively sense each channel in our sequence
    for i, actual_state in enumerate(ground_truth):
        pred, hits = sense_single_channel(strategy, actual_state, p_loss, n_window, tau) #sensing os whether a single channel is free or occupied and iterating it
        predictions.append(pred) #add the prediction of each channel into the array created
        all_hits.append(hits) #add number of hits used per channel to the array
        
        status = "OCCUPIED" if actual_state == 1 else "FREE"
        result_symbol = "✓" if pred == actual_state else "✗ (ERROR)" #if pred= actual then tick else cross
        print(f"Channel {i}: Ground Truth={status:<8} | Hits={hits:>2}/{n_window} | Sensor Guess={'OCCUPIED' if pred==1 else 'FREE':<8} {result_symbol}")
        
    # 4. Final Comparison Report
    print("\n--- Final Performance Summary ---")
    print(f"Ground Truth: {ground_truth}")
    print(f"Sensor Guess: {predictions}")
    
    correct_guesses = sum(1 for p, g in zip(predictions, ground_truth) if p == g) #if predicted is = ground truth=> correct
    accuracy = (correct_guesses / len(ground_truth)) * 100 #percentage error based on how much the predicted and ground truth match
    print(f"Sensing Accuracy: {accuracy:.1f}% ({correct_guesses}/{len(ground_truth)} correct)")


if __name__ == "__main__":
    # Define your pre-defined channel occupancy sequence (0 = free, 1 = occupied)
    channels_to_test = np.zeros(n_channels, dtype=int)
    channels_to_test[np.random.choice(n_channels, 5, replace=False)] = 1 
    print(channels_to_test)
    
    # Run the simulation
    run_spectrum_sensing_simulation(
        ground_truth=channels_to_test,
        strategy="quantum",   # Switch to "classical" to compare how the classical probe behaves!
        snr_db=-6.0,
        n_window=30,          # How many pulses to send down each channel
        target_pfa=0.05
    )

[0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0]
--- Starting Active Sensing Session (QUANTUM strategy) ---
Sensing SNR: -6.0 dB (Effective p_loss: 0.799)
Pulses per channel (N): 30 | Max allowed false alarm rate (Pfa): 0.05

[Calibration] Empty-channel success rate (p0): 0.242
[Calibration] Calculated threshold (tau): Declare Occupied if hits >= 12 out of 30

Channel 0: Ground Truth=FREE     | Hits= 9/30 | Sensor Guess=FREE     ✓
Channel 1: Ground Truth=FREE     | Hits=10/30 | Sensor Guess=FREE     ✓
Channel 2: Ground Truth=FREE     | Hits=10/30 | Sensor Guess=FREE     ✓
Channel 3: Ground Truth=FREE     | Hits= 4/30 | Sensor Guess=FREE     ✓
Channel 4: Ground Truth=FREE     | Hits= 7/30 | Sensor Guess=FREE     ✓
Channel 5: Ground Truth=FREE     | Hits= 9/30 | Sensor Guess=FREE     ✓
Channel 6: Ground Truth=FREE     | Hits= 7/30 | Sensor Guess=FREE     ✓
Channel 7: Ground Truth=FREE     | Hits= 8/30 | Sensor Guess=FREE     ✓
Channel 8: Ground Truth=FREE     | Hits= 8/30 | 

Generate random spectrum occupancy map
                │
                ▼
Convert SNR into depolarizing probability
                │
                ▼
Build quantum/classical sensing circuit
                │
                ▼
Create noisy communication channel
                │
                ▼
Calibrate detector on a known empty channel
                │
                ▼
Estimate baseline probability (p₀)
                │
                ▼
Calculate decision threshold (τ)
                │
                ▼
Sense each channel using N probe pulses
                │
                ▼
Count successful measurements (hits)
                │
                ▼
Compare hits with threshold τ
                │
                ▼
Predict Occupied or Free
                │
                ▼
Repeat for all channels
                │
                ▼
Compare predictions with ground truth
                │
                ▼
Compute and display sensing accuracy